## 3 - Run baseline deep learning (flat/macro)
In questo esperimento andrò a creare una baseline con reti LSTM sia su dataset flat che macro

In [1]:
import os
import pandas as pd
import sys


sys.path.append(os.path.abspath("../.."))
from utils.dl_helpers import prepare_sequential_data, evaluate_dl_model

In [2]:
DATASETS = {
    'flat': "/notebooks/data/processed",
    'macro': "/notebooks/data/processed_macro"
}

OUTPUT_DIR = "/notebooks/outputs/runs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:
#LAGS = [7, 14, 30, 60, 90] # Già runnati
LAGS = [360] # Extra, 360 giorni

for ds_name, input_dir in DATASETS.items():
    print(f"[LOG] STARTING LSTM EXPERIMENTS FOR DATASET: {ds_name.upper()}")
    
    for current_lag in LAGS:
        print(f"\n[LOG] Starting runs for LAG -> {current_lag} days")
        results_list = []
        
        for file_name in os.listdir(input_dir):
            if not file_name.endswith('.csv'):
                continue
                
            file_path = os.path.join(input_dir, file_name)
            
            # Pulizia nome corso
            course_name = file_name.replace('processed_', '').replace('macro_', '').replace('.csv', '')
            print(f"  -> Processing course: {course_name}")
            
            # 1. Preparazione Dati (Tensori 3D)
            X_3d, y, feature_names = prepare_sequential_data(file_path, lag=current_lag)
            
            if X_3d.shape[0] < 10:  # Salta corsi minuscoli che romperebbero la CV
                print(f"      [SKIP] Not enough students ({X_3d.shape[0]}) in {course_name}.")
                continue
                
            print(f"      Shape input 3D: {X_3d.shape}, Target: {y.shape}")
            
            # 2. Addestramento e Valutazione PyTorch (LSTM)
            metrics = evaluate_dl_model(
                X=X_3d, 
                y=y, 
                epochs=50, 
                batch_size=16, 
                lr=1e-3, 
                n_splits=10, 
                hidden_dim=100
            )
            
            if metrics is not None:
                result_row = {
                    'course': course_name,
                    'algorithm': 'LSTM',
                    'accuracy': round(metrics['accuracy'], 4),
                    'precision': round(metrics['precision'], 4),
                    'recall': round(metrics['recall'], 4),
                    'f1': round(metrics['f1'], 4),
                    'roc_auc': round(metrics['roc_auc'], 4),
                    'pr_auc': round(metrics['pr_auc'], 4)
                }
                results_list.append(result_row)
                
        # 3. Salvataggio risultati per il LAG e DATASET corrente
        if results_list:
            df_results = pd.DataFrame(results_list)
            output_csv = os.path.join(OUTPUT_DIR, f"baseline_{ds_name}_lag{current_lag}_LSTM_metrics.csv")
            df_results.to_csv(output_csv, index=False)
            print(f"\n[SUCCEEDED] Saved results in {output_csv}")
        else:
            print(f"\n[WARNING] No results generated for {ds_name} at LAG={current_lag}")

[LOG] STARTING LSTM EXPERIMENTS FOR DATASET: FLAT

[LOG] Starting runs for LAG -> 360 days
  -> Processing course: timeseries_13
      [SKIP] Not enough students (7) in timeseries_13.
  -> Processing course: timeseries_2
      Shape input 3D: (250, 360, 97), Target: (250,)
  -> Processing course: timeseries_11
      Shape input 3D: (539, 360, 97), Target: (539,)
  -> Processing course: timeseries_7
      Shape input 3D: (204, 360, 97), Target: (204,)
[WARNING] Classi sbilanciate (0: 6, 1: 198). Skipped.
  -> Processing course: timeseries_4
      Shape input 3D: (676, 360, 97), Target: (676,)
  -> Processing course: timeseries_3
      Shape input 3D: (75, 360, 97), Target: (75,)
[WARNING] Classi sbilanciate (0: 1, 1: 74). Skipped.
  -> Processing course: timeseries_5
      Shape input 3D: (269, 360, 97), Target: (269,)
  -> Processing course: timeseries_1
      Shape input 3D: (113, 360, 97), Target: (113,)
  -> Processing course: timeseries_6
      Shape input 3D: (469, 360, 97), Targe